## Program 18: Simple FOPC parser for logical expressions

Input: change `expression`. Supports predicates, AND, OR, NOT, and parentheses.

In [ ]:
import re

def tokenize(expression):
    return re.findall(r'[A-Za-z_][A-Za-z_0-9]*|[(),]', expression)

class FOPCParser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def peek(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def consume(self):
        token = self.peek()
        if token is not None:
            self.pos += 1
        return token

    def parse_atom(self):
        if self.peek() == "NOT":
            self.consume()
            return ("NOT", self.parse_atom())

        if self.peek() == "(":
            self.consume()
            result = self.parse_or()
            if self.consume() != ")":
                raise ValueError("Missing closing parenthesis")
            return result

        name = self.consume()
        if name is None or name in ("AND", "OR", "NOT", ",", ")"):
            raise ValueError("Expected predicate name")

        if self.peek() == "(":
            self.consume()
            args = []
            while self.peek() != ")":
                arg = self.consume()
                if arg is None or arg in ("AND", "OR", "NOT", ",", "("):
                    raise ValueError("Invalid predicate argument")
                args.append(arg)
                if self.peek() == ",":
                    self.consume()
                elif self.peek() != ")":
                    raise ValueError("Expected comma or closing parenthesis")
            self.consume()
            return ("PREDICATE", name, args)
        return ("SYMBOL", name)

    def parse_and(self):
        left = self.parse_atom()
        while self.peek() == "AND":
            self.consume()
            left = ("AND", left, self.parse_atom())
        return left

    def parse_or(self):
        left = self.parse_and()
        while self.peek() == "OR":
            self.consume()
            left = ("OR", left, self.parse_and())
        return left

    def parse(self):
        result = self.parse_or()
        if self.peek() is not None:
            raise ValueError("Unexpected token: " + self.peek())
        return result

expression = "Human(Socrates) AND Mortal(Socrates)"
parser = FOPCParser(tokenize(expression))
print("Expression:", expression)
print("Parsed structure:", parser.parse())